# 15 · Fine-tuning vs Prompting vs RAG

**Idea:** three ways to make a model 'know' your stuff:
- **Prompting**: put instructions/examples in the prompt. Cheapest, instant, limited by context.
- **RAG**: retrieve relevant text at query time and inject it. Great for *facts/knowledge* that change.
- **Fine-tuning**: retrain weights on your data. Best for *behavior/style/format*; costly, static.

Here we build a full RAG pipeline over `notes/` using `RagStore`. Fine-tuning stays conceptual (out of Phase 1 scope).

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'study_buddy').exists())
sys.path.insert(0, str(root))

from study_buddy import get_provider, RagStore

store = RagStore()
n = store.add_directory()  # indexes notes/*.md and *.txt
print('indexed files:', n, '| chunks:', len(store.chunks))

In [ ]:
query = 'Why do LLMs hallucinate and how does RAG help?'
for hit in store.search(query, k=3):
    print(f'[{hit.score:.3f}] ({hit.source}) {hit.text[:90]}...')

In [ ]:
# RAG answer: retrieve context, then ground the model on it.
context = store.context_for(query, k=3)
llm = get_provider('ollama')
messages = [
    {'role': 'system', 'content': 'Answer using ONLY the context. Cite the [source]. If missing, say so.'},
    {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
]
print(llm.chat(messages, temperature=0).text)

**Decision guide**

| Need | Use |
|------|-----|
| A few instructions/examples | Prompting |
| Large, changing knowledge base | RAG |
| Consistent tone/format/behavior | Fine-tuning |
| Private facts you must cite | RAG |

**Takeaway:** start with prompting, add RAG for knowledge, reach for fine-tuning last. You've now covered all 15 Phase 1 topics — the capstone ties them together.

Next → `capstone.ipynb` or run the WebUI: `streamlit run webui/app.py`